# PHM2009 — portable CPU experiment runner
Upload the updated Project folder to Drive, including existing Step 4–5 inputs and `result/test/colab_migration/path_migration.json`. Run cells in order. Choose CPU runtime. Existing validated results remain read-only; new outputs live under `result/test/colab_runs` and a separate Drive output directory.

Select `verify` first. `step4` executes the locked final pipeline for 50 Hz / High only; it does not repeat parameter sensitivity. `step5` executes the locked robustness dataset. Both reuse hash-validated prior recording results by default. Step 6 can be added once its scientific protocol is specified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title Experiment selection
DRIVE_PROJECT = '/content/drive/MyDrive/PHM2009/Project' # @param {type:'string'}
DRIVE_RESULTS = '/content/drive/MyDrive/PHM2009/execution_results' # @param {type:'string'}
STEP = 'verify' # @param ['verify', 'benchmark', 'step4', 'step5']
RUN_ID = 'default' # @param {type:'string'}
MAX_RECORDINGS = 0 # @param {type:'integer'}
REUSE_VALIDATED = True # @param {type:'boolean'}
GENERATE_STEP5_ANALYSIS = False # @param {type:'boolean'}
# MAX_RECORDINGS=0 means all selected recordings; positive means an execution batch.


## Copy inputs to local runtime
Source/data are read from Drive, never moved/deleted. Copy only required helical states, metadata and historical reference artifacts. Initial copying can take time. On a fresh runtime repeat this cell, then resume the same RUN_ID. Drive output commits are restored automatically by the runner.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, time, json
SOURCE = Path(DRIVE_PROJECT)
PROJECT = Path('/content/phm2009/Project')
DESTINATION = Path(DRIVE_RESULTS) / STEP / RUN_ID
assert SOURCE.is_dir(), f'Project missing: {SOURCE}'
assert not Path(DRIVE_RESULTS).resolve().is_relative_to(SOURCE.resolve()), 'Use a separate generated-output folder'
PROJECT.mkdir(parents=True, exist_ok=True)
start = time.perf_counter()
ignore = shutil.ignore_patterns('__pycache__', '_work', '*.tmp', '.git')
for relative in ['code', 'execution', 'result/test/step4', 'result/test/step4_corrected', 'result/test/step5', 'result/test/colab_migration']:
    shutil.copytree(SOURCE / relative, PROJECT / relative, dirs_exist_ok=True, ignore=ignore)
for name in ['requirements.txt', 'colab_runner.ipynb']:
    shutil.copy2(SOURCE / name, PROJECT / name)
for state in ['1', '2', '5', '6']:
    relative = 'DATA/labeled/helical ' + state
    shutil.copytree(SOURCE / relative, PROJECT / relative, dirs_exist_ok=True, ignore=ignore)
shutil.copy2(SOURCE / 'DATA/labeled/VidData.xls', PROJECT / 'DATA/labeled/VidData.xls')
os.chdir(PROJECT)
for variable in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[variable] = '1'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.path.insert(0, str(PROJECT))
print('Initial Drive -> runtime I/O seconds:', time.perf_counter() - start)


## Dependencies
Install the pinned numerical environment. All computations run in fresh Python subprocesses, so notebook imports made before installation do not affect the experiment. GCC is optional; compile/load/parity failures use NumPy safely. The saved Windows DLLs are historical hash-verified inputs and are never loaded on Linux.


In [ ]:
import venv
assert sys.version_info >= (3, 12), 'Pinned SciPy requires Python >=3.12'
ENVIRONMENT = Path('/content/phm2009/environment')
venv.EnvBuilder(with_pip=True).create(ENVIRONMENT)
PYTHON = str(ENVIRONMENT / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
print('Isolated Python:', PYTHON, 'Compiler:', shutil.which('g++'))


## Verify inputs and run tests
Any genuinely missing/changed dependency stops execution. Verification does not regenerate a lock to hide changes. Tests use separate output artifacts. A Linux native library must pass numerical parity before it is enabled.


In [ ]:
subprocess.run([PYTHON, '-B', '-m', 'execution.runner', '--step', 'verify'], check=True)
subprocess.run([PYTHON, '-B', 'result/test/step5/scripts/tests.py'], check=True)
subprocess.run([PYTHON, '-B', '-m', 'execution.tests'], check=True)


## Run or resume
Checkpoints are committed after both channels of each full-length recording, with per-channel recovery for a partly completed recording. Arrays and metadata are SHA256-verified before skip. Drive copies write payloads before completion markers. Keep RUN_ID to resume; change it only for a separate run. Reused historical arrays remain referenced in the uploaded project.

Full Step 5 analysis is optional and uses the original analysis/report code. Enable it only for a complete Step 5 inventory (MAX_RECORDINGS=0). Statistical analysis may take additional time.


In [ ]:
command = [PYTHON, '-B', '-m', 'execution.runner', '--step', STEP, '--run-id', RUN_ID]
if STEP in ['step4', 'step5']:
    command += ['--drive-output', str(DESTINATION)]
    if MAX_RECORDINGS > 0: command += ['--max-recordings', str(MAX_RECORDINGS)]
    if not REUSE_VALIDATED: command += ['--recompute']
    if GENERATE_STEP5_ANALYSIS:
        assert STEP == 'step5' and MAX_RECORDINGS == 0
        command += ['--analyze']
subprocess.run(command, check=True)


## Final output sync
Recording commits already persist during experiments. This cell also saves verification/tests and benchmark outputs. Benchmark reports CPU/wall time, sampled peak RSS and local I/O separately; compare the Colab report with the Windows baseline rather than assuming a speedup.


In [ ]:
# Use a subprocess so numerical imports also use the isolated environment.
sys.path.insert(0, str(PROJECT))
from execution.provenance import ROOT, sha
# Sync helper imports only stdlib until an experiment is initialized.
from execution.runner import sync_tree
start = time.perf_counter()
if STEP == 'benchmark':
    sync_tree(PROJECT / 'result/test/colab_runs/benchmark' / RUN_ID, DESTINATION)
if STEP in ['step4', 'step5']:
    sync_tree(PROJECT / 'result/test/colab_runs' / STEP / RUN_ID, DESTINATION)
sync_tree(PROJECT / 'result/test/colab_migration', Path(DRIVE_RESULTS) / 'environment_audit')
print('Final runtime -> Drive I/O seconds:', time.perf_counter() - start)
print('Drive outputs:', DESTINATION)


Google documents transient runtime lifetime and recommends limiting Drive I/O by copying inputs locally: [Colab FAQ](https://research.google.com/colaboratory/faq.html). This runner uses CPU NumPy/SciPy; choosing GPU does not accelerate these algorithms.
